In [ ]:
# Main working folder

# Task 1: File ingestion and regular expressions

# 1. write a robust file reader function
import re
from pathlib import Path


def read_file(file_path: Path) -> tuple[str, str]:
    """
    Function reads files, and returns a tuple that 
    contains both the content of the file (as strings), as 
    well as the encoding that is used. The function is able to 
    handle both UTF-8 and iso-8859-1 encoding. 
    
    Args: Path object that points the file that is supposed
    to be read. 
    Returns: text file, encoding
    """
    try: 
        text = file_path.read_text(encoding="utf-8")
        return text, "utf-8"
    except UnicodeDecodeError:
        text = file_path.read_text(encoding="iso-8859-1")
        return text, "iso-8859-1"


# 2. write a log parser function


def parser_func(log_content: str) -> list[dict]:
    """
    Function looks through lines, finding timestamp, register_id, 
    item, price, currency, and status while ignoring
    noice lines. Function looks for square brackets (/[, /]),
    creates labeled slots with the appropriate name (?P<"name">),
    and pulls out the item inside (.*?) Everything is collected
    in a dictorinary. The items are put into the correct format
    according to what they are. 

    Args: 
    Returns: 
    """

    valid_lines = (
        r"\[(?P<timestamp>.*?)\] "  
      r"\[(?P<register_id>.*?)\] "  
      r"item=(?P<item>\w+) " # words with one or more letters/numbers
      r"price=(?P<price>[\d.]+) " # digits, one or more
      r"(?P<currency>[A-Z]{3}) " # big letters from A-Z, three letters. "EUR" as currency
      r"\((?P<status>\w+)\)"
    )


    transaction_record = []

    # turn groups above into dictionaries before
    # adding to transaction record
    for match in re.finditer(valid_lines, log_content):
        convert_dictionary = match.groupdict()
        convert_dictionary["price"]=float(convert_dictionary["price"])
        transaction_record.append(convert_dictionary)

    return transaction_record

# Testing the code
test = Path("Data/2026/library_cafe/POS_01/register_sales.log")

text, code = read_file(test)
transactions = parser_func(text)

print(f"Encoding: {code}")
print(f"Amount of transactions: {len(transactions)}")
print(f"First transaction: {transactions}")
  



Encoding: utf-8
Amount of transactions: 80
First transaction: [{'timestamp': '2026-10-15 07:00:42', 'register_id': 'POS_01', 'item': 'Sandwich', 'price': 7.33, 'currency': 'USD', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:02:21', 'register_id': 'POS_01', 'item': 'Sandwich', 'price': 7.57, 'currency': 'USD', 'status': 'REFUND'}, {'timestamp': '2026-10-15 07:03:54', 'register_id': 'POS_01', 'item': 'Sandwich', 'price': 7.1, 'currency': 'USD', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:04:08', 'register_id': 'POS_01', 'item': 'Sandwich', 'price': 7.9, 'currency': 'NOK', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:06:38', 'register_id': 'POS_01', 'item': 'Cappuccino', 'price': 999.0, 'currency': 'NOK', 'status': 'CANCELLED'}, {'timestamp': '2026-10-15 07:07:00', 'register_id': 'POS_01', 'item': 'Cappuccino', 'price': 4.25, 'currency': 'NOK', 'status': 'SUCCESS'}, {'timestamp': '2026-10-15 07:09:23', 'register_id': 'POS_01', 'item': 'Croissant', 'price': 4.04, 'curren

In [18]:
# Task 2: Recursive utility functions 

import json
from pathlib import Path

# 1. resursive directory search function
def find_files(directory: str|Path) -> list[Path]:
    """
    Function finds all log files in a nested directionary. Uses Path.iterdir()
    to go through items. 
    Path.glob(), Path.rglob(), os.walk() are not used. 

    Args: 
    Returns: list of pathlib.Path objects.
    """

    dir = Path(directory)
    log_file = []

    if dir.is_dir():
        for item in dir.iterdir():
            if item.is_dir(): 
                log_file.extend(find_files(item))
            elif item.is_file() and item.suffix == ".log":
                log_file.append(item)

    return log_file

# 2. Combination of nested dictionaries and lists. Returns register id strings. 
def active_register_id(data: dict|list) -> set[str]:
    """
    Function goes through dictionaries and lists found in store_config.json. 
    Function finds all registered id strings that are active. 
    
    Args: 
    Returns: a set of all resgister id strings that are active. 
    """

    active_register=set()
    if isinstance(data, dict): #if data is a dictionary

        if "id" in data and data.get("active") is True:
            active_register.add(str(data["id"]))
        for value in data.values():
           active_register.update(active_register_id(value))
    elif isinstance(data, list): #if data is a list
        for item in data: 
            active_register.update(active_register_id(item))

    return active_register


# Testing the code: 
file = list(Path(".").rglob("store_config.json"))
if not file:
  file = list(Path("..").rglob("store_config.json"))

if file:
  config_file = file[0]
  test_folder = config_file.parent

  # Task 2.1 Test
  log_file = find_files(test_folder)
  print(f"Amount of .log-filer: {len(log_file)}")
  if log_file:
    print(f"First log-file: {log_file[0]}")

  # Task 2.2 Test
  with open(config_file) as f:
    json_data = json.load(f)

  active = active_register_id(json_data)
  print(f"Active registers: {active}")
  print(f"Amount of active registers: {len(active)}")

  

Amount of .log-filer: 9
First log-file: Data/2026/science_building/POS_01/register_sales.log
Active registers: {'POS_01', 'POS_03', 'POS_02'}
Amount of active registers: 3
